In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import

In [32]:
!pip install torch
!pip install tensorflow
!pip install sentence-transformers

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sentence_transformers import SentenceTransformer

# Input

In [ ]:
text_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/NLP entire process practice/data source/text_data.csv")

In [ ]:
documents = text_data.iloc[:500].copy()
#documents = text_data.copy()

documents

,citationStringAnnotated
0,"['author', 'family', 'ritchie', 'family', 'giv..."
1,"['author', 'family', 'hwang', 'family', 'given..."
2,"['author', 'family', 'malson', 'family', 'give..."
3,"['author', 'family', 'hoang', 'family', 'given..."
4,"['anon', 'issued', 'year', 'year', 'issued', '..."
...,...
495,"['author', 'family', 'sharma', 'family', 'give..."
496,"['author', 'family', 'simmons', 'family', 'giv..."
497,"['author', 'family', 'flynn', 'family', 'given..."
498,"['author', 'family', 'chen', 'family', 'given'..."


In [ ]:
def safe_literal_eval(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        # Return an empty list for malformed strings to prevent errors
        return []

# 1. Safely convert the strings back into real Python lists
# Apply safe_literal_eval to the specific column 'citationStringAnnotated'
documents['citationStringAnnotated'] = documents['citationStringAnnotated'].apply(safe_literal_eval)

# 2. Now standard join works perfectly!
# Apply the join to the elements of the lists in the same specific column
documents['citationStringAnnotated'] = documents['citationStringAnnotated'].apply(lambda x: " ".join(x))

# Update 'documents' to be a Series of processed strings for direct use with TfidfVectorizer
documents = documents['citationStringAnnotated']

documents

,citationStringAnnotated
0,author family ritchie family given e given and...
1,author family hwang family given li san given ...
2,author family malson family given bruce a give...
3,author family hoang family given andré h given...
4,anon issued year year issued title acknowledgm...
...,...
495,author family sharma family given deepika give...
496,author family simmons family given scott c giv...
497,author family flynn family given frank t given...
498,author family chen family given huiwen given a...


In [ ]:
documents = documents.tolist()

# TF - IDF:

In [ ]:
# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    min_df=5,                  # Drops words that appear in fewer than 5 documents (removes weird typos like 'aa')
    max_df=0.90,               # Drops words that appear in >90% of documents (acts as a dynamic stop-word filter)
    max_features=2000,         # Keeps ONLY the top 2000 most important words (shrinks your matrix)
    token_pattern=r'(?u)\b[A-Za-z]+\b'  # ONLY keeps alphabetical words (drops numbers, underscores, and foreign symbols)
)

tfidf_matrix = vectorizer.fit_transform(documents)

# Fit and transform the documents
tfidf_matrix = vectorizer.fit_transform(documents)

# Get feature names (words) from the vectorizer
feature_names = vectorizer.get_feature_names_out()

# Convert the TF-IDF matrix to a dense array and display the results
dense_array = tfidf_matrix.toarray()
print("TF-IDF Matrix:")
print(dense_array)

# Display feature names
print("\nFeature Names:")
print(feature_names)

TF-IDF Matrix:
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.22473212 0.         0.         ... 0.         0.         0.        ]
 ...
 [0.15858876 0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]

Feature Names:
['a' 'abstract' 'acid' 'acta' 'after' 'al' 'ali' 'american' 'an'
 'analysis' 'and' 'anon' 'applications' 'applied' 'as' 'assessment'
 'association' 'at' 'author' 'b' 'based' 'be' 'berlin' 'between' 'bf'
 'biology' 'book' 'bulletin' 'by' 'c' 'canadian' 'cancer' 'carbon' 'care'
 'case' 'cell' 'ch' 'characteristics' 'characterization' 'charles'
 'chemical' 'chemistry' 'chen' 'children' 'chin' 'clinical' 'co'
 'computer' 'conference' 'd' 'data' 'david' 'de' 'der' 'des' 'design'
 'determination' 'development' 'dictionary' 'd

In [ ]:
# Convert to a dataframe just to look at the clean column names
clean_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
clean_df.head()

,a,abstract,acid,acta,after,al,ali,american,an,analysis,...,wang,water,williams,with,world,x,y,yang,yin,z
0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.270427,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.224732,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Statistics Summarize -- Dianogsis:

In [ ]:
# 1. Sum the TF-IDF scores for every single column (word) across all 500 documents
# This compresses the matrix into a single row of total scores
sum_words = tfidf_matrix.sum(axis=0)

# 2. Match those total scores back to the actual text words
words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]

# 3. Sort the list from mathematically highest score to lowest
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

# 4. View the mathematical outliers (The top 10-20 are usually your structural noise)
print("Top 15 Mathematically Heaviest Words:")
for word, score in words_freq[:15]:
    print(f"{word}: {score:.2f}")

Top 15 Mathematically Heaviest Words:
family: 148.91
given: 148.56
page: 72.38
volume: 71.79
issue: 71.57
author: 68.86
of: 46.11
and: 42.56
publisher: 35.62
pp: 34.44
the: 33.34
in: 31.46
a: 25.19
s: 24.70
anon: 22.90


# TF - IDF rebuild:

In [ ]:
# 1. Take the metadata words we mathematically discovered
custom_noise = [
    'family', 'given', 'page', 'volume', 'issue',
    'author', 'publisher', 'pp', 'anon', 's'
]

# 2. Combine them with standard English stop words ('the', 'and', 'a', etc.)
final_stop_words = list(ENGLISH_STOP_WORDS) + custom_noise

# 3. Put this ultimate list into your Vectorizer
vectorizer = TfidfVectorizer(
    stop_words=final_stop_words,
    min_df=2,
    max_df=0.85, # Lowered slightly to be safe!
    max_features=1000,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b'
)

tfidf_matrix = vectorizer.fit_transform(documents)

# Fit and transform the documents
tfidf_matrix = vectorizer.fit_transform(documents)

# Get feature names (words) from the vectorizer
feature_names = vectorizer.get_feature_names_out()

# Convert the TF-IDF matrix to a dense array and display the results
dense_array = tfidf_matrix.toarray()
print("TF-IDF Matrix:")
print(dense_array)

# Display feature names
print("\nFeature Names:")
print(feature_names)

TF-IDF Matrix:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Feature Names:
['abscisic' 'absorption' 'abstract' 'abstracts' 'abu' 'academy' 'acid'
 'acm' 'acoustic' 'acprof' 'acs' 'acta' 'activity' 'acute'
 'administration' 'advanced' 'advances' 'aeronautics' 'affairs' 'african'
 'age' 'aged' 'ahmed' 'aic' 'aiche' 'aid' 'akira' 'al' 'algorithm' 'ali'
 'alkyl' 'allen' 'alloy' 'alloys' 'als' 'america' 'american' 'amm'
 'analysis' 'angewandte' 'animal' 'anisotropy' 'ann' 'annals' 'annual'
 'anomalous' 'anti' 'antonio' 'aorta' 'aortic' 'application'
 'applications' 'applied' 'approach' 'approaches' 'approximation' 'arch'
 'archiv' 'armor' 'art' 'artery' 'article' 'artificial' 'artists' 'arts'
 'asce' 'aspects' 'assembly' 'assessment' 'associated' 'association'
 'astronautics' 'asymmetric' 'atomic' 'australian' 'bacteria'
 'bacteriology' 'bakar' 'base' 'based' 'bay' 'behavior' 'beh

In [ ]:
# Convert to a dataframe just to look at the clean column names
clean_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
clean_df.head()

,abscisic,absorption,abstract,abstracts,abu,academy,acid,acm,acoustic,acprof,...,youth,yu,zealand,zeitschrift,zhang,zhao,zhou,zhu,zone,zur
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Statistics Summarize -- Dianogsis:

In [ ]:
# 1. Sum the TF-IDF scores for every single column (word) across all 500 documents
# This compresses the matrix into a single row of total scores
sum_words = tfidf_matrix.sum(axis=0)

# 2. Match those total scores back to the actual text words
words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]

# 3. Sort the list from mathematically highest score to lowest
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

# 4. View the mathematical outliers (The top 10-20 are usually your structural noise)
print("Top 15 Mathematically Heaviest Words:")
for word, score in words_freq[:15]:
    print(f"{word}: {score:.2f}")

Top 15 Mathematically Heaviest Words:
journal: 23.40
et: 12.73
al: 11.99
science: 9.56
und: 8.21
international: 7.50
press: 7.36
american: 7.02
university: 6.77
review: 6.11
springer: 5.96
john: 5.73
bf: 5.51
ieee: 5.32
der: 5.26


# SVD

In [ ]:
# Choose your 'k' (number of latent topics)
k = 50
svd_model = TruncatedSVD(n_components=k)

# This works directly on the sparse matrix without crashing!
A_k = svd_model.fit_transform(tfidf_matrix)

In [ ]:
print(np.shape(A_k))
display(A_k)

(500, 50)


array([[ 4.42633687e-02, -3.78582727e-03,  1.47537502e-02, ...,
        -1.35942366e-01,  1.36107142e-02,  5.39771688e-05],
       [ 1.11689059e-01, -1.27718457e-02,  3.63898349e-04, ...,
         5.73514400e-03, -5.11098197e-03,  4.98332808e-02],
       [ 2.30225688e-02,  2.72688008e-03,  3.09873524e-03, ...,
         8.80332685e-02, -1.09369829e-02, -6.42427264e-02],
       ...,
       [ 4.87291962e-02,  2.70660713e-03,  2.01207725e-02, ...,
        -1.26659569e-01,  1.09943646e-01, -5.84371310e-02],
       [ 1.50388120e-01, -1.10050707e-02,  8.02686768e-03, ...,
        -5.96672809e-03,  1.34846134e-01, -6.80730892e-02],
       [ 6.98693801e-02, -1.31885976e-02,  6.14361795e-03, ...,
         6.16824981e-02,  3.40040443e-02,  8.03403229e-02]])

# Saving

In [ ]:
# 1. Generate column names based on the number of topics (k=50)
# This creates a list: ['Topic_0', 'Topic_1', 'Topic_2' ... 'Topic_49']
topic_labels = [f"Topic_{i}" for i in range(A_k.shape[1])]

# 2. Convert the A_k numpy array into a DataFrame
df_svd = pd.DataFrame(A_k, columns=topic_labels)

# 3. View the clean DataFrame
display(df_svd.head())

,Topic_0,Topic_1,Topic_2,Topic_3,Topic_4,Topic_5,Topic_6,Topic_7,Topic_8,Topic_9,...,Topic_40,Topic_41,Topic_42,Topic_43,Topic_44,Topic_45,Topic_46,Topic_47,Topic_48,Topic_49
0,0.044263,-0.003786,0.014754,0.032538,-0.010822,-0.070966,-0.069806,-0.070802,0.194661,0.020756,...,0.020670,-0.027263,-0.036304,0.011908,-0.054394,0.046122,0.088352,-0.135942,0.013611,0.000054
1,0.111689,-0.012772,0.000364,0.064527,0.013214,-0.153017,-0.114342,-0.112233,0.261470,-0.001002,...,-0.094314,-0.087410,0.001330,-0.081684,-0.105379,-0.006900,-0.069859,0.005735,-0.005111,0.049833
2,0.023023,0.002727,0.003099,0.015856,0.010834,-0.025910,-0.017561,0.015014,-0.012638,0.007165,...,0.001644,-0.117694,-0.182547,-0.088125,0.124328,0.192661,-0.015288,0.088033,-0.010937,-0.064243
3,0.154798,-0.005997,0.020020,0.153544,-0.058398,-0.044240,0.049793,0.042693,-0.103889,0.001431,...,0.046366,-0.056738,0.002101,-0.019977,-0.007389,0.213054,-0.086247,-0.155132,-0.057203,0.091707
4,0.014710,-0.003785,-0.001835,-0.008348,0.000795,-0.014367,0.004240,-0.006214,-0.005955,0.006704,...,-0.000072,0.089946,0.029541,0.049144,0.071864,-0.046507,0.098677,-0.014048,-0.012442,0.042254


In [ ]:
# 2. Save to CSV
df_svd.to_csv('/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/NLP entire process practice/data source/df_svd.csv', index=False)

# Phase 1: Finish what you started (The Baseline)

## Input

In [ ]:
A_k = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/NLP entire process practice/data source/df_svd.csv")
text_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/NLP entire process practice/data source/text_data.csv").iloc[:500].copy()

In [ ]:
def safe_literal_eval(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        # Return an empty list for malformed strings to prevent errors
        return []

documents = text_data.copy()

# 1. Safely convert the strings back into real Python lists
# Apply safe_literal_eval to the specific column 'citationStringAnnotated'
documents['citationStringAnnotated'] = documents['citationStringAnnotated'].apply(safe_literal_eval)

# 2. Now standard join works perfectly!
# Apply the join to the elements of the lists in the same specific column
documents['citationStringAnnotated'] = documents['citationStringAnnotated'].apply(lambda x: " ".join(x))

#documents = documents.transpose()
documents

,citationStringAnnotated
0,author family ritchie family given e given and...
1,author family hwang family given li san given ...
2,author family malson family given bruce a give...
3,author family hoang family given andré h given...
4,anon issued year year issued title acknowledgm...
...,...
495,author family sharma family given deepika give...
496,author family simmons family given scott c giv...
497,author family flynn family given frank t given...
498,author family chen family given huiwen given a...


In [ ]:
A_k

,Topic_0,Topic_1,Topic_2,Topic_3,Topic_4,Topic_5,Topic_6,Topic_7,Topic_8,Topic_9,...,Topic_40,Topic_41,Topic_42,Topic_43,Topic_44,Topic_45,Topic_46,Topic_47,Topic_48,Topic_49
0,0.044263,-0.003786,0.014754,0.032538,-0.010822,-0.070966,-0.069806,-0.070802,0.194661,0.020756,...,0.020670,-0.027263,-0.036304,0.011908,-0.054394,0.046122,0.088352,-0.135942,0.013611,0.000054
1,0.111689,-0.012772,0.000364,0.064527,0.013214,-0.153017,-0.114342,-0.112233,0.261470,-0.001002,...,-0.094314,-0.087410,0.001330,-0.081684,-0.105379,-0.006900,-0.069859,0.005735,-0.005111,0.049833
2,0.023023,0.002727,0.003099,0.015856,0.010834,-0.025910,-0.017561,0.015014,-0.012638,0.007165,...,0.001644,-0.117694,-0.182547,-0.088125,0.124328,0.192661,-0.015288,0.088033,-0.010937,-0.064243
3,0.154798,-0.005997,0.020020,0.153544,-0.058398,-0.044240,0.049793,0.042693,-0.103889,0.001431,...,0.046366,-0.056738,0.002101,-0.019977,-0.007389,0.213054,-0.086247,-0.155132,-0.057203,0.091707
4,0.014710,-0.003785,-0.001835,-0.008348,0.000795,-0.014367,0.004240,-0.006214,-0.005955,0.006704,...,-0.000072,0.089946,0.029541,0.049144,0.071864,-0.046507,0.098677,-0.014048,-0.012442,0.042254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.222021,0.034784,0.024708,-0.104378,-0.082444,0.035727,0.085646,-0.073333,-0.105096,-0.301795,...,-0.014390,0.023989,-0.024629,0.055493,-0.026275,-0.025225,-0.025720,-0.020750,-0.040419,0.025258
496,0.091086,-0.007486,0.000733,0.018157,0.029957,-0.147213,-0.050366,0.054842,-0.053300,0.033350,...,-0.066963,-0.060811,0.060892,-0.114663,0.080796,-0.007044,0.037472,0.072847,0.032770,-0.052660
497,0.048729,0.002707,0.020121,0.033066,0.030071,-0.025296,-0.036463,-0.033898,0.098942,0.034850,...,-0.048178,-0.040341,0.034099,-0.093258,-0.003688,0.168673,0.083539,-0.126660,0.109944,-0.058437
498,0.150388,-0.011005,0.008027,0.127103,-0.008537,-0.076925,0.012523,0.030847,-0.090839,-0.006675,...,0.104514,-0.055066,-0.029827,-0.100480,-0.038845,0.063598,-0.007176,-0.005967,0.134846,-0.068073


## Kmeans

In [ ]:
# 1. Choose the number of clusters (groups) you want to find.
# Let's assume you think there are roughly 5 main topics in your 500 documents.
num_clusters = 5

# 2. Initialize the K-Means algorithm
# 'random_state=42' ensures that if you run this cell 10 times, you get the exact same result.
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

# 3. Fit the model to your SVD matrix and assign a cluster to every document
print("Running K-Means...")
cluster_labels = kmeans.fit_predict(A_k)
print("Done!")

# 4. The output is just an array of numbers (e.g., [0, 2, 1, 0...]).
# Let's attach those numbers back to your original DataFrame so it makes sense!
# (Assuming your original dataframe is called 'df')
documents['Cluster'] = cluster_labels

# 5. See how many documents the algorithm put into each bucket
print("\nDocuments per cluster:")
print(documents['Cluster'].value_counts())

# 6. Look at the actual text of a few documents in Cluster 0 to see what they have in common
print("\n--- Sample Documents from Cluster 0 ---")
display(documents[documents['Cluster'] == 0][['citationStringAnnotated', 'Cluster']].head())

Running K-Means...
Done!

Documents per cluster:
Cluster
3    389
4     59
2     22
1     22
0      8
Name: count, dtype: int64

--- Sample Documents from Cluster 0 ---


,citationStringAnnotated,Cluster
74,anon issued year year issued title contents ti...,0
156,author family ottevanger family given p b give...,0
187,author family zhang family given jianqiao give...,0
223,author family aderajew family given tamirat s ...,0
247,author family yuce family given c given author...,0


## Dianogsis

In [ ]:
# 1. Reverse the K-Means center points back through the SVD model to the TF-IDF word space
original_space_centroids = svd_model.inverse_transform(kmeans.cluster_centers_)
# 2. Get the actual vocabulary words from your Vectorizer
feature_names = vectorizer.get_feature_names_out()
# 3. Print the top 5 defining words for every single cluster
print("--- REAL TOPICS PER CLUSTER ---")
for i in range(num_clusters): # Assuming num_clusters is still 5
    print(f"\nCluster {i} Theme:")

    # Sort the centroid's weights to find the heaviest, most important words for this group
    top_word_indices = original_space_centroids[i].argsort()[-5:][::-1]
    top_words = [feature_names[ind] for ind in top_word_indices]

    print(", ".join(top_words))

--- REAL TOPICS PER CLUSTER ---

Cluster 0 Theme:
journal, european, physical, tb, critical

Cluster 1 Theme:
und, der, bf, klinische, wochenschrift

Cluster 2 Theme:
vi, elsevier, design, research, dan

Cluster 3 Theme:
journal, science, press, american, international

Cluster 4 Theme:
et, al, journal, cancer, patients


# Phase 2: The "Bridge" (DNN with your current data)

## DNN model

In [27]:
# ---------------------------------------------------------
# 1. Prepare the Data (Convert to PyTorch Tensors)
# ---------------------------------------------------------
X = A_k  # Your 50-column SVD matrix
y = documents['Cluster'].values # The 5 clusters (0, 1, 2, 3, 4)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert Numpy arrays into PyTorch Tensors
# X needs to be floats, y needs to be integers (LongTensors)
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# ---------------------------------------------------------
# 2. Build the Neural Network Architecture
# ---------------------------------------------------------
class ClusterPredictor(nn.Module):
    def __init__(self):
        super(ClusterPredictor, self).__init__()
        # Layer 1: 50 Inputs -> 64 Neurons
        self.layer1 = nn.Linear(50, 64)
        # Dropout to prevent memorizing the data
        self.dropout = nn.Dropout(0.2)
        # Layer 2: 64 Neurons -> 32 Neurons
        self.layer2 = nn.Linear(64, 32)
        # Output Layer: 32 Neurons -> 5 Outputs (for the 5 clusters)
        self.output_layer = nn.Linear(32, 5)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        # Note: PyTorch's loss function applies Softmax automatically,
        # so we just return the raw numbers (logits) here!
        x = self.output_layer(x)
        return x

# Initialize the model
model = ClusterPredictor()

# ---------------------------------------------------------
# 3. Define the Engine (Loss Function and Optimizer)
# ---------------------------------------------------------
# CrossEntropyLoss is perfect for multi-class classification
criterion = nn.CrossEntropyLoss()
# Adam optimizer updates the weights based on the errors
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ---------------------------------------------------------
# 4. The Training Loop (The PyTorch Way)
# ---------------------------------------------------------
epochs = 20

print("Starting Training...\n")
for epoch in range(epochs):
    # Set model to training mode
    model.train()

    # Step A: Forward Pass (Make a guess)
    predictions = model(X_train_tensor)

    # Step B: Calculate the Error (Loss)
    loss = criterion(predictions, y_train_tensor)

    # Step C: Backward Pass (Calculate how to fix the error)
    optimizer.zero_grad() # Clear old gradients
    loss.backward()       # Do the math
    optimizer.step()      # Update the weights

    # --- Check accuracy on unseen Test Data every 5 epochs ---
    if (epoch + 1) % 5 == 0:
        model.eval() # Set model to evaluation mode
        with torch.no_grad(): # Turn off gradient calculation to save memory
            test_preds = model(X_test_tensor)
            # Find the cluster with the highest score
            _, predicted_classes = torch.max(test_preds, 1)
            # Calculate accuracy
            correct = (predicted_classes == y_test_tensor).sum().item()
            accuracy = correct / y_test_tensor.size(0)

            print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Validation Accuracy: {accuracy * 100:.2f}%")

print("\nTraining Complete!")

Starting Training...

Epoch 5/20 | Loss: 1.2119 | Validation Accuracy: 78.00%
Epoch 10/20 | Loss: 0.7898 | Validation Accuracy: 78.00%
Epoch 15/20 | Loss: 0.7190 | Validation Accuracy: 78.00%
Epoch 20/20 | Loss: 0.5670 | Validation Accuracy: 78.00%

Training Complete!


# Phase 3: True Deep Learning (Drop the SVD)

## Rebuild Kmeans with BERT

In [38]:
# 1. Load a pre-trained Deep Learning language model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Define the exact words that are ruining the sentence structure
words_to_remove = ['author', 'family', 'given', 'issued', 'year', 'title', 'container', 'publisher', 'anon']

# 2. Write a quick function to delete them from the string
def clean_deep_text(text):
    for word in words_to_remove:
        # Replace the noise words with empty space
        text = text.replace(f"{word} ", "")
    return text

# 3. Apply it to your data
documents['Cleaned_String'] = documents['citationStringAnnotated'].apply(clean_deep_text)

# 4. NOW feed the completely clean, natural sentences to BERT!
print("Calculating deep meanings...")
deep_embeddings = embedder.encode(documents['Cleaned_String'].tolist())

# 5. Cluster these rich deep-learning vectors instead of the shallow TF-IDF matrix
kmeans_deep = KMeans(n_clusters=5, random_state=42)
documents['Deep_Cluster'] = kmeans_deep.fit_predict(deep_embeddings)

# 6. View the results
print(documents['Deep_Cluster'].value_counts())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Calculating deep meanings...
Deep_Cluster
2    161
0    122
3     77
1     76
4     64
Name: count, dtype: int64


## Hidden NLP

In [39]:
# ==========================================
# STEP 1: Modern Text Preparation (No TF-IDF!)
# ==========================================
# We don't count words anymore. We just assign every unique word an ID number.
# Example: "author" = 1, "science" = 2, "cancer" = 3
tokenizer = Tokenizer(num_words=5000) # Keep top 5000 words
tokenizer.fit_on_texts(documents['citationStringAnnotated'])

# Convert the text into sequences of numbers: "science cancer" -> [2, 3]
X_seq = tokenizer.texts_to_sequences(documents['citationStringAnnotated'])

# Neural networks need uniform sizes. We pad short sentences with 0s so every row is exactly 100 words long.
max_length = 100
X_padded = pad_sequences(X_seq, maxlen=max_length, padding='post')

vocab_size = len(tokenizer.word_index) + 1 # Total unique words

# Prepare the PyTorch Tensors
y = documents['Deep_Cluster'].values
X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# ==========================================
# STEP 2: The Modern Architecture
# ==========================================
class ModernNLP(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_size=64):
        super(ModernNLP, self).__init__()

        # UPGRADE 1: The Embedding Layer (Replaces SVD)
        # This creates a dynamic, trainable "dictionary of meaning" for your words.
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)

        # UPGRADE 2: The LSTM (Reads the text sequentially)
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_size, batch_first=True)

        # Final prediction layer (5 outputs for your 5 clusters)
        self.fc = nn.Linear(hidden_size, 5)

    def forward(self, x):
        # 1. Convert Word IDs into rich meaning vectors
        x = self.embedding(x)

        # 2. Pass vectors into the LSTM.
        # The LSTM outputs the sequence, but we only care about the very last hidden state
        # because it contains the "memory" of the entire sentence!
        lstm_out, (hidden, cell) = self.lstm(x)
        final_memory_state = hidden[-1]

        # 3. Make the final prediction
        out = self.fc(final_memory_state)
        return out

# Initialize the modern model
model = ModernNLP(vocab_size=vocab_size)

# ==========================================
# STEP 3: The Engine & Training Loop
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005) # Slightly slower learning rate for Embeddings

epochs = 20

print("Starting Deep Learning Training...\n")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    predictions = model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            test_preds = model(X_test_tensor)
            _, predicted_classes = torch.max(test_preds, 1)
            correct = (predicted_classes == y_test_tensor).sum().item()
            accuracy = correct / y_test_tensor.size(0)

            print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Validation Accuracy: {accuracy * 100:.2f}%")

Starting Deep Learning Training...

Epoch 5/20 | Loss: 1.5464 | Validation Accuracy: 33.00%
Epoch 10/20 | Loss: 1.5467 | Validation Accuracy: 33.00%
Epoch 15/20 | Loss: 1.5463 | Validation Accuracy: 33.00%
Epoch 20/20 | Loss: 1.5435 | Validation Accuracy: 33.00%


## Dianogsis

In [40]:
# Print 3 sample documents from each of the 5 new Deep Clusters
for i in range(5):
    print(f"\n=========================================")
    print(f"       SAMPLES FROM DEEP CLUSTER {i}      ")
    print(f"=========================================")

    # Get the text for this specific cluster
    cluster_docs = documents[documents['Deep_Cluster'] == i]['citationStringAnnotated'].head(3)

    for idx, text in enumerate(cluster_docs):
        # We print the first 150 characters of the citation so it's easy to read
        print(f"{idx + 1}. {text[:150]}...\n")


       SAMPLES FROM DEEP CLUSTER 0      
1. author family hwang family given li san given and family divoky family given david given author issued year year issued title breaking wave setup and ...

2. author family hoang family given andré h given family plätzer family given simon given and family samitz family given daniel given author issued year ...

3. author family benge family given glen given family darby family given john given family peroyea family given miles given family aguilar family given t...


       SAMPLES FROM DEEP CLUSTER 1      
1. author family malson family given bruce a given author issued year year issued title tarnished armor erosion of military ethics title publisher defens...

2. author family karari family given peter given family byrne family given sean given family skarlato family given olga given and family ahmed family giv...

3. anon issued year year issued title conclusion title in container title japanese and chinese immigrant activists container t